# Static Stats

Produce basic stats for static GNSS testing, after removing the first 15 minutes (warm up) and last 2 minutes (shut down).

Copyright 2024 Michael George (AKA Logiqx).

This file is part of [GPS Wizard](https://github.com/Logiqx/gps-wizard) and is distributed under the terms of the GNU General Public License.

GPS Wizard is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version.

GPS Wizard is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with GPS Wizard. If not, see <https://www.gnu.org/licenses/>.

## Notes

This is very crude and simply looks for Doppler speeds in excess of 30 m/s (i.e. 108 km/h or 58.32 knots).

Such speeds are currently unrealistic for a windsurfer.

In [1]:
import os
import sys

import numpy as np

import traceback

corePath = os.path.join('..', 'core')
if corePath not in sys.path:
    sys.path.extend([corePath])

from file_reader import getFileReader

## Main Function

In [2]:
WARM_UP_SECS = 1800
SHUTDOWN_SECS = 300

MS_TO_KNOTS = 3600 / 1852

def getInterval(data):
    '''Determine the typical interval (milliseconds) between timestamps'''

    tsDiffs = ((data['ts'][1:] - data['ts'][:-1]) * 1000).round().astype(int)
    uniqueValues, counts = np.unique(tsDiffs, return_counts=True)
    interval = uniqueValues[np.argmax(counts)]
    
    return interval


def main():
    '''Iterate through sessions testing each GPS file'''

    rootDir = os.path.join(projdir, '..', 'ubx-cfg', 'gpslogs', 'syrac', 'test-8')

    errors = {}
    spikes = {}
    
    totFiles = 0
    totPoints = 0
    totDist = 0

    print('dir,file,points,rate,min,max,mean,stddev,pc50,pc68,pc95,pc99.7')
    
    for root, subDirs, files in os.walk(rootDir):
        for file in sorted(files):
            ext = os.path.splitext(file)[1].lower()

            # Only interested in some file types
            if ext and ext in ['.gpy', '.sbp', '.oao']:
                filePath = os.path.join(root, file)
                reader = getFileReader(filePath)
                try:
                    reader.load()
                    
                    # Process all tracks within the file
                    for track in reader.tracks:
                        interval = getInterval(track.data)
                        rate = round(1000 / interval)

                        minTs = np.min(track.data['ts'])
                        maxTs = np.max(track.data['ts'])
                        
                        points = (np.argwhere((track.data['ts'] >= minTs + WARM_UP_SECS) & (track.data['ts'] <= maxTs - SHUTDOWN_SECS)))

                        # e.g. sog, sat, or ehve
                        metric = 'sog'

                        minVal = np.min(track.data[metric][points]) * MS_TO_KNOTS
                        maxVal = np.max(track.data[metric][points]) * MS_TO_KNOTS
                        meanVal = np.mean(track.data[metric][points]) * MS_TO_KNOTS
                        stdVal = np.std(track.data[metric][points]) * MS_TO_KNOTS
                        percentiles = np.percentile(track.data[metric][points], [50, 68, 95, 99.7]) * MS_TO_KNOTS

                        print('{},{},{},{},{:.3f},{:.3f},{:.5f},{:.5f},{:.5f},{:.5f},{:.5f},{:.5f}'.format(
                            os.path.basename(root), file, len(points), rate, minVal, maxVal, meanVal, stdVal,
                            percentiles[0], percentiles[1], percentiles[2], percentiles[3]))

                    totFiles += 1

                except Exception:
                    errors[filePath.replace(projdir + '/', '')] = traceback.format_exc()
                    print('E', end='')

    print(os.linesep * 2 + 'Summary: files = {}'.format(totFiles))

    if len(errors) > 0:
        print(os.linesep * 2 + 'Errors:')
        for error in errors:
            print(error)
            print(errors[error])

In [3]:
if __name__ == '__main__':
    projdir = os.path.realpath(os.path.join(sys.path[0], "..", ".."))

    main()
    
    print(os.linesep + 'All done!')

dir,file,points,rate,min,max,mean,stddev,pc50,pc68,pc95,pc99.7
test-8,SYRAC_E2_26-11-01_08-55_5Hz.gpy,97822,5,0.000,0.150,0.02721,0.01543,0.02527,0.03305,0.05637,0.08553
test-8,SYRAC_E2_26-11-01_14-59_8Hz.gpy,156492,8,0.000,0.229,0.04186,0.02274,0.03888,0.05054,0.08359,0.12246
test-8,SYRAC_E2_26-11-02_08-55_8Hz.gpy,160130,8,0.000,0.224,0.04176,0.02290,0.03888,0.05054,0.08359,0.12441
test-8,SYRAC_E2_26-11-02_15-07_5Hz.gpy,96842,5,0.000,0.156,0.02858,0.01639,0.02527,0.03305,0.05832,0.09136
test-8,SYRAC_E6_26-11-01_08-55_8Hz.gpy,156572,8,0.000,0.266,0.05004,0.02719,0.04665,0.06026,0.10108,0.14579
test-8,SYRAC_E6_26-11-01_14-59_5Hz.gpy,97841,5,0.000,0.342,0.03157,0.01771,0.02916,0.03693,0.06415,0.09719
test-8,SYRAC_E6_26-11-02_08-55_5Hz.gpy,100081,5,0.000,0.177,0.03255,0.01834,0.02916,0.03888,0.06609,0.10108
test-8,SYRAC_E6_26-11-02_15-07_8Hz.gpy,154986,8,0.000,0.993,0.04909,0.02707,0.04471,0.05832,0.09914,0.14579


Summary: files = 8

All done!
